# LeetCode 1490 — Clone N-ary Tree bằng Python

Link đề: https://leetcode.com/problems/clone-n-ary-tree/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search

> **Ghi chú Premium:** đây là bài Premium 🔒. Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1490/). Nếu có tài khoản Premium, bạn nên đối chiếu lại đề gốc.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children if children is not None else []
"""

class Solution:
    def cloneTree(self, root: 'Node') -> 'Node':
        ...
```

## 1. Mục tiêu học

1. Hiểu **cây N-phân** (mỗi nút có danh sách con `children` dài tuỳ ý) và cách LeetCode tuần tự hoá nó theo level-order có `null` phân nhóm.
2. Phân biệt bản sao **nông** và **sâu**; đặc biệt bẫy riêng của Python: **danh sách `children` dùng chung** và **tham số mặc định có thể thay đổi** (`children=[]`).
3. Viết được 4 cách: tuần tự hoá rồi dựng lại, DFS đệ quy (cách của doocs), DFS lặp bằng stack, BFS bằng hàng đợi cặp `(cũ, mới)`.
4. Biết cách tổng quát hoá cho **đồ thị** (follow-up của đề) bằng `dict` ánh xạ nút cũ → nút mới.
5. Viết hàm kiểm tra bản sao sâu: không nút/không danh sách nào trùng `id()` với cây gốc, cấu trúc và giá trị khớp.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây N-phân, trả về **bản sao sâu** (deep copy) của cây. Mỗi nút có `val` (số nguyên) và `children` (danh sách các nút con).

**Tuần tự hoá:** theo level-order; **mỗi nhóm con** của một nút được ngăn bởi `null`. Ví dụ `[1,null,3,2,4,null,5,6]`: gốc 1; `null`; con của 1 là `3,2,4`; `null`; con của 3 là `5,6`; (con của 2, 4, 5, 6 rỗng — các `null` cuối bị lược bỏ).

**Ví dụ 1:** `root = [1,null,3,2,4,null,5,6]` → `[1,null,3,2,4,null,5,6]`

**Ví dụ 2:** `root = [1,null,2,3,4,5,null,null,6,7,null,8,null,9,10,null,null,11,null,12,null,13,null,null,14]` → giống input.

**Ràng buộc:** độ sâu cây `≤ 1000`; tổng số nút trong `[0, 10^4]`.

**Follow-up:** lời giải có dùng được cho bài **đồ thị** không?

Output luôn trùng input khi tuần tự hoá — cái khó là mọi nút (và mọi danh sách `children`) phải là đối tượng **mới**.

In [1]:
import copy
import random
import sys
import time
from collections import deque
from typing import List, Optional

class Node:
    def __init__(self, val=None, children=None):
        self.val = val
        self.children = children if children is not None else []
    def __repr__(self):
        return f"Node({self.val})"

ex1 = [1, None, 3, 2, 4, None, 5, 6]
ex2 = [1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None,
       11, None, 12, None, 13, None, None, 14]
examples = [ex1, ex2]
for e in examples:
    print(e)

[1, None, 3, 2, 4, None, 5, 6]
[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]


Hai hàm phụ tự dựng để test:
- `build_nary(data)`: hàng đợi các nút chờ nhận con; đọc lần lượt từng nhóm (dừng ở `None`).
- `serialize(root)`: ghi `[root.val, None]`, rồi với mỗi nút theo BFS ghi các con và một `None`; cuối cùng bỏ `None` thừa.

In [2]:
def build_nary(data):
    if not data:
        return None
    root = Node(data[0])
    q = deque([root])
    i = 2                              # bỏ qua root và dấu None sau root
    while q and i <= len(data):
        cur = q.popleft()
        while i < len(data) and data[i] is not None:
            ch = Node(data[i])
            cur.children.append(ch)
            q.append(ch)
            i += 1
        i += 1                         # bỏ qua None kết thúc nhóm
    return root

def serialize(root):
    if root is None:
        return []
    out = [root.val, None]
    q = deque([root])
    while q:
        cur = q.popleft()
        for ch in cur.children:
            out.append(ch.val)
            q.append(ch)
        out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

for e in examples:
    print(serialize(build_nary(e)) == e)

True
True


## 3. Kiến thức nền

### 3.1 Cây N-phân trong Python

Mỗi nút là một đối tượng có `val` và `children` (một `list` các nút). Duyệt các con bằng `for ch in node.children`.

In [3]:
root = Node(1, [Node(3, [Node(5), Node(6)]), Node(2), Node(4)])
def show(node, depth=0):
    print("  " * depth + str(node.val))
    for ch in node.children:
        show(ch, depth + 1)
show(root)

1
  3
    5
    6
  2
  4


### 3.2 `is`, `id()` và danh sách dùng chung

Gán `b = a.children` không sao chép danh sách: `b` và `a.children` là **cùng một list**. Sửa `b` ⇒ cây gốc bị sửa theo. Muốn list mới: `list(a.children)` hoặc `a.children[:]` (nhưng các phần tử bên trong vẫn là nút cũ — đó mới là sao chép nông một tầng).

In [4]:
a = Node(1, [Node(2)])
b_list = a.children                 # cùng một list
c_list = list(a.children)           # list mới, nhưng phần tử vẫn là nút cũ
b_list.append(Node(99))
print("a.children:", a.children)
print("c_list is a.children:", c_list is a.children, "| c_list[0] is a.children[0]:", c_list[0] is a.children[0])

a.children: [Node(2), Node(99)]
c_list is a.children: False | c_list[0] is a.children[0]: True


### 3.3 Bẫy kinh điển: tham số mặc định có thể thay đổi

`def f(x=[])`: list `[]` được tạo **một lần** khi định nghĩa hàm và **dùng chung** cho mọi lần gọi. Vì thế định nghĩa `Node` đúng phải dùng `children=None` rồi tạo list mới bên trong — đúng như skeleton của đề.

In [5]:
class BadNode:
    def __init__(self, val=None, children=[]):     # nguy hiểm!
        self.val = val
        self.children = children

x, y = BadNode(1), BadNode(2)
x.children.append("con của x")
print("y.children =", y.children, "| dùng chung list?", x.children is y.children)
p, q2 = Node(1), Node(2)
print("Node đúng: dùng chung list?", p.children is q2.children)

y.children = ['con của x'] | dùng chung list? True
Node đúng: dùng chung list? False


### 3.4 List comprehension và đệ quy

`[f(ch) for ch in node.children]` áp dụng `f` cho từng con, tạo một **list mới** — rất hợp để dựng `children` cho nút sao chép.

In [6]:
def count_nodes(node):
    return 1 + sum(count_nodes(ch) for ch in node.children)
def vals_doubled(node):
    return [node.val * 2] + [v for ch in node.children for v in vals_doubled(ch)]
t = build_nary(ex1)
print("số nút:", count_nodes(t), "| giá trị x2 (tiền thứ tự):", vals_doubled(t))

số nút: 6 | giá trị x2 (tiền thứ tự): [2, 6, 10, 12, 4, 8]


### 3.5 BFS bằng `deque`, DFS bằng stack; giới hạn đệ quy

`deque.popleft()` là `O(1)`. Cây sâu tới 1000 tầng có thể chạm giới hạn đệ quy mặc định (~1000) ⇒ khi test ta nâng `sys.setrecursionlimit`, hoặc dùng bản lặp.

In [7]:
q = deque([build_nary(ex2)]); order = []
while q:
    x = q.popleft(); order.append(x.val); q.extend(x.children)
print("BFS ví dụ 2:", order)
print("recursion limit mặc định:", sys.getrecursionlimit())

BFS ví dụ 2: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
recursion limit mặc định: 1000


## 4. Ý tưởng nền tảng / trực giác

Sao chép tay **Ví dụ 1** (gốc 1, con `3, 2, 4`; 3 có con `5, 6`):

1. Muốn có `1'` (bản sao của 1) ta cần `children` của `1'` = `[3', 2', 4']`.
2. Muốn có `3'` cần `[5', 6']`; `5'`, `6'` là lá ⇒ tạo ngay.
3. `2'`, `4'` là lá.

Đây là **định nghĩa đệ quy**: *bản sao của một nút = nút mới cùng giá trị, với danh sách con mới gồm bản sao của từng con.* Vì là **cây** (mỗi nút có đúng một cha, không chu trình), mỗi nút được sao đúng một lần, **không cần** `dict` ghi nhớ như bài có con trỏ `random` (LC 1485).

| nút | con (gốc) | con (bản sao) |
|---|---|---|
| 5 | [] | [] |
| 6 | [] | [] |
| 3 | [5, 6] | [5', 6'] |
| 2 | [] | [] |
| 4 | [] | [] |
| 1 | [3, 2, 4] | [3', 2', 4'] |

(Thứ tự hoàn thành là hậu thứ tự: con xong trước, cha sau.)

In [8]:
t = build_nary(ex1)
c5, c6 = Node(5), Node(6)
c3 = Node(3, [c5, c6]); c2 = Node(2); c4 = Node(4)
c1 = Node(1, [c3, c2, c4])
print("tự tay:", serialize(c1), "| khớp?", serialize(c1) == serialize(t), "| là nút mới?", c1 is not t)

tự tay: [1, None, 3, 2, 4, None, 5, 6] | khớp? True | là nút mới? True


## 5. Cách 1 — Brute Force / baseline: tuần tự hoá rồi dựng lại

**Ý tưởng:** tận dụng hai hàm sẵn có: `serialize(root)` ra mảng, rồi `build_nary(mảng)` dựng cây mới. Cây dựng lại gồm toàn nút mới.

Đây là "baseline" theo nghĩa **dễ nghĩ ra**, không phải chậm: vẫn `O(n)`, nhưng tốn thêm một mảng trung gian, phải viết hai hàm, và chỉ dùng được khi giá trị không cần gì khác ngoài `val`. Một baseline khác là `copy.deepcopy(root)` — đúng ở đây vì lớp đầu ra cũng là `Node`, nhưng phỏng vấn không chấp nhận và `deepcopy` đệ quy rất sâu (dễ tràn với cây 1000 tầng).

In [9]:
def clone_roundtrip(root):
    return build_nary(serialize(root))

def clone_deepcopy(root):
    return copy.deepcopy(root)

for e in examples:
    t = build_nary(e)
    a, b = clone_roundtrip(t), clone_deepcopy(t)
    print(serialize(a) == e, a is not t, "|", serialize(b) == e, b is not t)

True True | True True
True True | True True


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho mảng trung gian + hàng đợi.

**Nút thắt / hạn chế:** đi hai lượt và cấp phát mảng trung gian không cần thiết; nếu nút có thêm trường khác (ví dụ con trỏ `parent`, `random`) thì định dạng tuần tự hoá phải mở rộng. Ta muốn sao chép **trực tiếp** trong một lượt.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** cây có cấu trúc đệ quy ⇒ bản sao cũng định nghĩa đệ quy:

`clone(None) = None`; `clone(x) = Node(x.val, [clone(c) for c in x.children])`.

**Vì sao không cần memo?** Trong cây, mỗi nút chỉ có **một** cha ⇒ chỉ được gặp **một lần** khi đi từ gốc ⇒ không thể sao trùng. (Đồ thị thì khác — xem Cách 5.)

**Quy tắc "nếu… thì…":**
- Nếu `root is None` thì trả `None`.
- Nếu nút là lá thì bản sao có `children = []` — **list mới**, không phải `root.children`.
- Nếu muốn tránh đệ quy: dùng stack/queue chứa **cặp** `(nút cũ, nút mới)`; lấy cặp ra, với mỗi con cũ tạo con mới, gắn vào `new.children`, rồi đẩy cặp con vào cấu trúc.

**Invariant cho bản lặp:** mọi cặp `(old, new)` trong stack/queue thoả `new.val == old.val`, `new` đã được gắn vào đúng vị trí trong cây mới, nhưng `new.children` **chưa** được điền.

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy (cách của doocs)

Một dòng then chốt: `children = [self.cloneTree(child) for child in root.children]`, rồi `return Node(root.val, children)`. List comprehension tạo **list mới** chứa **nút mới**.

In [10]:
def clone_rec(root):
    if root is None:
        return None
    return Node(root.val, [clone_rec(ch) for ch in root.children])

for e in examples:
    t = build_nary(e); c = clone_rec(t)
    print(serialize(c) == e, c is not t)

True True
True True


Code nộp LeetCode (Cách 2, đúng như doocs):

In [11]:
class SolutionRec:
    def cloneTree(self, root: 'Node') -> 'Node':
        if root is None:
            return None
        children = [self.cloneTree(child) for child in root.children]
        return Node(root.val, children)

print(serialize(SolutionRec().cloneTree(build_nary(ex1))))

[1, None, 3, 2, 4, None, 5, 6]


**Độ phức tạp:** `O(n)` thời gian (mỗi nút tạo một lần, mỗi cạnh duyệt một lần), `O(h)` bộ nhớ ngăn xếp đệ quy (`h ≤ 1000` là độ sâu).

### Cách 3 — DFS lặp bằng stack các cặp `(cũ, mới)`

1. Tạo `new_root = Node(root.val)`, đẩy `(root, new_root)`.
2. Lấy cặp ra; với mỗi con `c` của `old` (theo đúng thứ tự): tạo `nc = Node(c.val)`, `new.children.append(nc)`, đẩy `(c, nc)`.

Thứ tự lấy ra khỏi stack không quan trọng, vì mỗi con mới đã được gắn **đúng vị trí** ngay khi tạo.

In [12]:
def clone_stack(root):
    if root is None:
        return None
    new_root = Node(root.val)
    st = [(root, new_root)]
    while st:
        old, new = st.pop()
        for c in old.children:
            nc = Node(c.val)
            new.children.append(nc)
            st.append((c, nc))
    return new_root

for e in examples:
    print(serialize(clone_stack(build_nary(e))) == e)

True
True


Code nộp LeetCode (Cách 3):

In [13]:
class SolutionStack:
    def cloneTree(self, root: 'Node') -> 'Node':
        if root is None:
            return None
        new_root = Node(root.val)
        st = [(root, new_root)]
        while st:
            old, new = st.pop()
            for c in old.children:
                nc = Node(c.val)
                new.children.append(nc)
                st.append((c, nc))
        return new_root

print(serialize(SolutionStack().cloneTree(build_nary(ex2))))

[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10, None, None, 11, None, 12, None, 13, None, None, 14]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ stack tệ nhất (nút có rất nhiều con). Không lo giới hạn đệ quy.

### Cách 4 — BFS bằng hàng đợi cặp `(cũ, mới)`

Giống Cách 3 nhưng dùng `deque` và `popleft()` ⇒ sao chép **theo từng tầng**. Hữu ích khi muốn kết hợp thêm thông tin theo tầng (độ sâu, …).

In [14]:
def clone_bfs(root):
    if root is None:
        return None
    new_root = Node(root.val)
    q = deque([(root, new_root)])
    while q:
        old, new = q.popleft()
        new.children = [Node(c.val) for c in old.children]
        q.extend(zip(old.children, new.children))
    return new_root

class SolutionBFS:
    def cloneTree(self, root: 'Node') -> 'Node':
        if root is None:
            return None
        new_root = Node(root.val)
        q = deque([(root, new_root)])
        while q:
            old, new = q.popleft()
            new.children = [Node(c.val) for c in old.children]
            q.extend(zip(old.children, new.children))
        return new_root

for e in examples:
    print(serialize(clone_bfs(build_nary(e))) == e, serialize(SolutionBFS().cloneTree(build_nary(e))) == e)

True True
True True


`zip(old.children, new.children)` ghép từng con cũ với con mới tương ứng ⇒ đẩy cả loạt vào hàng đợi. **Độ phức tạp:** `O(n)` thời gian, `O(w)` bộ nhớ hàng đợi (`w` = độ rộng lớn nhất).

### Cách 5 — Tổng quát cho đồ thị (follow-up): `dict` ghi nhớ

Nếu một nút có thể được nhiều nút trỏ tới (đồ thị, DAG) hoặc có chu trình, cách đệ quy thuần sẽ **sao trùng** hoặc **lặp vô hạn**. Thêm `seen[old] = new`, ghi **trước** khi đệ quy (giống LC 133 Clone Graph). Với cây, cách này vẫn đúng, chỉ tốn thêm `dict`.

In [15]:
def clone_memo(root):
    seen = {}
    def dfs(old):
        if old is None:
            return None
        if old in seen:
            return seen[old]
        new = Node(old.val)
        seen[old] = new                      # ghi sổ trước khi xử lý con
        new.children = [dfs(c) for c in old.children]
        return new
    return dfs(root)

class SolutionMemo:
    def cloneTree(self, root: 'Node') -> 'Node':
        return clone_memo(root)

# Thử trên một "đồ thị" có chu trình: a -> b -> a
a, b = Node("a"), Node("b")
a.children.append(b); b.children.append(a)
ca = clone_memo(a)
print(ca.val, ca.children[0].val, ca.children[0].children[0] is ca, ca is not a)

a b True True


## 8. TRACE từng bước

Trace Cách 3 (stack) trên Ví dụ 1. Bảng trạng thái stack (ghi theo giá trị, `'` là nút mới):

| bước | lấy ra | con được tạo & gắn | stack sau bước |
|---|---|---|---|
| 0 | — | tạo 1' | [(1,1')] |
| 1 | (1,1') | 3', 2', 4' gắn vào 1' | [(3,3'),(2,2'),(4,4')] |
| 2 | (4,4') | — | [(3,3'),(2,2')] |
| 3 | (2,2') | — | [(3,3')] |
| 4 | (3,3') | 5', 6' gắn vào 3' | [(5,5'),(6,6')] |
| 5 | (6,6') | — | [(5,5')] |
| 6 | (5,5') | — | [] |

In [16]:
def trace_stack(data):
    root = build_nary(data)
    new_root = Node(root.val)
    st = [(root, new_root)]
    step = 0
    print(f"bước {step}: tạo {root.val}' | stack={[(o.val) for o, _ in st]}")
    while st:
        old, new = st.pop()
        step += 1
        made = []
        for c in old.children:
            nc = Node(c.val)
            new.children.append(nc)
            st.append((c, nc))
            made.append(f"{c.val}'")
        print(f"bước {step}: lấy ({old.val},{old.val}') | tạo & gắn {made} | stack={[o.val for o, _ in st]}")
    print("kết quả:", serialize(new_root))
    return new_root

_ = trace_stack(ex1)

bước 0: tạo 1' | stack=[1]
bước 1: lấy (1,1') | tạo & gắn ["3'", "2'", "4'"] | stack=[3, 2, 4]
bước 2: lấy (4,4') | tạo & gắn [] | stack=[3, 2]
bước 3: lấy (2,2') | tạo & gắn [] | stack=[3]
bước 4: lấy (3,3') | tạo & gắn ["5'", "6'"] | stack=[5, 6]
bước 5: lấy (6,6') | tạo & gắn [] | stack=[5]
bước 6: lấy (5,5') | tạo & gắn [] | stack=[]
kết quả: [1, None, 3, 2, 4, None, 5, 6]


Trace Cách 2 (đệ quy) để thấy thứ tự **vào** (tiền thứ tự) và **xong** (hậu thứ tự):

In [17]:
def trace_rec(node, depth=0):
    pad = "  " * depth
    print(f"{pad}vào {node.val}: có {len(node.children)} con")
    kids = [trace_rec(ch, depth + 1) for ch in node.children]
    new = Node(node.val, kids)
    print(f"{pad}xong {node.val}' với con {[k.val for k in kids]}")
    return new

c = trace_rec(build_nary(ex1))
print(serialize(c))

vào 1: có 3 con
  vào 3: có 2 con
    vào 5: có 0 con
    xong 5' với con []
    vào 6: có 0 con
    xong 6' với con []
  xong 3' với con [5, 6]
  vào 2: có 0 con
  xong 2' với con []
  vào 4: có 0 con
  xong 4' với con []
xong 1' với con [3, 2, 4]
[1, None, 3, 2, 4, None, 5, 6]


## 9. Code nộp LeetCode

Bản chính: DFS đệ quy (đúng như doocs). Trên LeetCode lớp `Node` đã có sẵn. Nếu lo cây sâu 1000 tầng, có thể nộp bản stack (Cách 3) — kết quả như nhau.

In [18]:
class Solution:
    def cloneTree(self, root: 'Node') -> 'Node':
        if root is None:
            return None
        children = [self.cloneTree(child) for child in root.children]
        return Node(root.val, children)

for e in examples:
    t = build_nary(e)
    c = Solution().cloneTree(t)
    assert serialize(c) == e and c is not t
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

Chứng minh Cách 2 bằng **quy nạp theo chiều cao** của cây con:

- **Cơ sở:** `root` là lá (không con) ⇒ trả `Node(root.val, [])`: nút mới, cùng giá trị, list con mới rỗng ⇒ là bản sao sâu.
- **Bước quy nạp:** giả sử `cloneTree(c)` trả bản sao sâu đúng của mọi cây con `c` (thấp hơn). Khi đó `children` là một **list mới**, phần tử thứ `i` là bản sao sâu của `root.children[i]` (giữ đúng thứ tự). `Node(root.val, children)` là nút mới cùng giá trị ⇒ cả cây là bản sao sâu.
- **Không trùng lặp:** các cây con của các con khác nhau là **rời nhau** (tính chất cây) ⇒ không nút nào được sao hai lần; mọi nút trong kết quả đều do `Node(...)` tạo mới.

Với bản lặp (Cách 3, 4): invariant ở mục 6 đảm bảo mỗi nút cũ được đưa vào cấu trúc đúng một lần (khi cha của nó được lấy ra), con mới được `append` theo đúng thứ tự `old.children` ⇒ cây mới đẳng cấu với cây cũ.

## 11. Phân tích Time / Space

Gọi `n` = số nút (`≤ 10^4`), `h` = độ sâu (`≤ 1000`), `w` = độ rộng lớn nhất.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Tuần tự hoá + dựng lại | `O(n)` | `O(n)` mảng trung gian |
| 1'. `copy.deepcopy` | `O(n)` | `O(n)` memo nội bộ + đệ quy sâu |
| 2. DFS đệ quy | `O(n)` | `O(h)` ngăn xếp đệ quy |
| 3. DFS stack | `O(n)` | `O(n)` tệ nhất |
| 4. BFS | `O(n)` | `O(w)` |
| 5. Memo (đồ thị) | `O(n + số cạnh)` | `O(n)` |

Không tính bộ nhớ của cây kết quả (bắt buộc `n` nút mới).

## 12. Bộ test

### 12.1 Hàm kiểm tra bản sao sâu

`check_deep_copy(orig, cp)`:
1. `serialize` bằng nhau (cấu trúc + giá trị + thứ tự con).
2. Không nút nào của bản sao trùng `id()` với nút của cây gốc.
3. Không **list `children`** nào của bản sao là list của cây gốc.
4. Số nút bằng nhau.

In [19]:
def all_nodes(root):
    out, st = [], [root] if root else []
    while st:
        x = st.pop(); out.append(x); st.extend(x.children)
    return out

def check_deep_copy(orig, cp):
    if orig is None or cp is None:
        return (orig is None and cp is None), "None"
    if serialize(orig) != serialize(cp):
        return False, "cấu trúc/giá trị khác"
    o, c = all_nodes(orig), all_nodes(cp)
    if len(o) != len(c):
        return False, "số nút khác"
    if {id(x) for x in o} & {id(x) for x in c}:
        return False, "dùng chung nút với cây gốc"
    if {id(x.children) for x in o} & {id(x.children) for x in c}:
        return False, "dùng chung list children"
    return True, "OK"

t = build_nary(ex1)
print(check_deep_copy(t, Solution().cloneTree(t)), check_deep_copy(t, t))

(True, 'OK') (False, 'dùng chung nút với cây gốc')


### 12.2 Ví dụ đề + edge case

In [20]:
sys.setrecursionlimit(10000)
ALL = {
    "roundtrip": clone_roundtrip,
    "deepcopy": clone_deepcopy,
    "rec": clone_rec,
    "stack": clone_stack,
    "bfs": clone_bfs,
    "memo": clone_memo,
    "sub_rec": lambda r: SolutionRec().cloneTree(r),
    "sub_stack": lambda r: SolutionStack().cloneTree(r),
    "sub_bfs": lambda r: SolutionBFS().cloneTree(r),
    "sub_memo": lambda r: SolutionMemo().cloneTree(r),
    "submit": lambda r: Solution().cloneTree(r),
}
edge = [
    [],                                   # cây rỗng
    [7],                                  # một nút
    [1, None, 2],                         # một con
    [1, None, 1, 1, 1, 1],                # giá trị trùng
    [1, None, 2, None, 3, None, 4],       # dây xích
    [0, None] + list(range(1, 51)),       # gốc có 50 con
]
for name, f in ALL.items():
    for data in examples + edge:
        orig = build_nary(data)
        before = serialize(orig)
        cp = f(orig)
        ok, why = check_deep_copy(orig, cp)
        assert ok, (name, data, why)
        assert serialize(orig) == before
print("Mọi cách đúng với", len(examples + edge), "test; cây gốc không đổi")

Mọi cách đúng với 8 test; cây gốc không đổi


### 12.3 Cây ngẫu nhiên (seed cố định)

Sinh cây `n` nút: nút thứ `i` chọn cha ngẫu nhiên trong các nút trước đó (thiên về nút gần để có cả cây sâu lẫn cây rộng).

In [21]:
def random_nary(rng, n, maxv=10):
    if n == 0:
        return None
    nodes = [Node(rng.randint(0, maxv))]
    for i in range(1, n):
        if rng.random() < 0.5:
            par = nodes[rng.randrange(max(0, i - 3), i)]
        else:
            par = nodes[rng.randrange(i)]
        nd = Node(rng.randint(0, maxv))
        par.children.append(nd)
        nodes.append(nd)
    return nodes[0]

rng = random.Random(1490)
for _ in range(500):
    orig = random_nary(rng, rng.randint(0, 60))
    for name, f in ALL.items():
        ok, why = check_deep_copy(orig, f(orig))
        assert ok, (name, why)
print("500 cây ngẫu nhiên: mọi cách cho bản sao sâu hợp lệ")

500 cây ngẫu nhiên: mọi cách cho bản sao sâu hợp lệ


### 12.4 Giới hạn: `10^4` nút, và dây xích sâu 1000

In [22]:
def chain(n):
    root = cur = Node(0)
    for v in range(1, n):
        nxt_node = Node(v); cur.children.append(nxt_node); cur = nxt_node
    return root

big = random_nary(rng, 10**4, 10**4)
deep = chain(1000)
for label, orig in [("10^4 nút", big), ("xích 1000", deep)]:
    for name in ["roundtrip", "rec", "stack", "bfs", "memo", "submit"]:
        t0 = time.perf_counter(); cp = ALL[name](orig); dt = time.perf_counter() - t0
        assert check_deep_copy(orig, cp)[0]
        print(f"{label:10} | {name:9}: {dt*1000:7.2f} ms")

10^4 nút   | roundtrip:    9.15 ms
10^4 nút   | rec      :   31.54 ms
10^4 nút   | stack    :    4.23 ms
10^4 nút   | bfs      :    8.35 ms
10^4 nút   | memo     :    6.64 ms
10^4 nút   | submit   :    5.15 ms
xích 1000  | roundtrip:    1.01 ms
xích 1000  | rec      :    0.40 ms
xích 1000  | stack    :    0.27 ms
xích 1000  | bfs      :    0.50 ms
xích 1000  | memo     :    0.60 ms
xích 1000  | submit   :    0.26 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng lại list `children` của cây gốc

`Node(root.val, root.children)` tạo nút gốc mới nhưng **toàn bộ con vẫn là nút cũ** (và chung list). Sửa bản sao ⇒ hỏng bản gốc.

In [23]:
def wrong1(root):
    if root is None:
        return None
    return Node(root.val, root.children)       # SAI: sao chép nông

orig = build_nary(ex1)
bad = wrong1(orig)
print("serialize giống?", serialize(bad) == serialize(orig), "| kiểm tra:", check_deep_copy(orig, bad))
bad.children.append(Node(100))                 # sửa bản sao...
print("cây gốc sau khi sửa bản sao:", serialize(orig))
orig = build_nary(ex1)
print("đúng:", check_deep_copy(orig, Solution().cloneTree(orig)))

serialize giống? True | kiểm tra: (False, 'dùng chung nút với cây gốc')
cây gốc sau khi sửa bản sao: [1, None, 3, 2, 4, 100, None, 5, 6]
đúng: (True, 'OK')


Chỉ nhìn `serialize` thì tưởng đúng! Phải kiểm tra `id()`. **Sửa:** tạo list mới chứa **bản sao** của từng con.

### Lỗi 2 — Sao chép từng nút nhưng gán chung list con

Viết `new = Node(old.val); new.children = old.children; ...rồi sửa new.children[i] = clone(...)` — thao tác gán phần tử **ghi đè vào list của cây gốc**.

In [24]:
def wrong2(root):
    if root is None:
        return None
    new = Node(root.val)
    new.children = root.children               # SAI: chung list
    for i, ch in enumerate(root.children):
        new.children[i] = wrong2(ch)           # ghi đè luôn vào cây gốc!
    return new

orig = build_nary(ex1)
keep = all_nodes(orig)
bad = wrong2(orig)
print("kiểm tra:", check_deep_copy(orig, bad))
print("con đầu của gốc CŨ có còn là nút cũ không?", any(orig.children[0] is k for k in keep))

kiểm tra: (False, 'dùng chung nút với cây gốc')
con đầu của gốc CŨ có còn là nút cũ không? False


Cây gốc bị "thay ruột" bằng nút mới — bản gốc và bản sao dùng chung list ⇒ checker báo lỗi. **Sửa:** `new.children = [clone(c) for c in old.children]` (list mới).

### Lỗi 3 — Định nghĩa `Node` với `children=[]`

Khi tự viết lớp `Node` để test (hoặc ở bài tương tự tự định nghĩa lớp), dùng tham số mặc định `[]` ⇒ **mọi nút tạo không truyền `children` đều chung một list** ⇒ bản sao (và cả cây test) loạn.

In [25]:
class NodeBad:
    def __init__(self, val=None, children=[]):   # SAI
        self.val = val
        self.children = children

def clone_with(cls, root):
    if root is None:
        return None
    new = cls(root.val)                          # không truyền children
    for ch in root.children:
        new.children.append(clone_with(cls, ch))
    return new

orig = build_nary([1, None, 2, 3])
bad = clone_with(NodeBad, orig)
print("số con của nút 2 trong bản sao:", len(bad.children[0].children), "(mong đợi 0)")
print("con 2 có tự chứa chính nó?", bad.children[0].children is bad.children)
good = clone_with(Node, orig)
print("với Node đúng:", serialize(good))

số con của nút 2 trong bản sao: 2 (mong đợi 0)
con 2 có tự chứa chính nó? True
với Node đúng: [1, None, 2, 3]


Mọi `NodeBad` dùng chung **một** list: gốc và các con cùng "nhìn" thấy một danh sách ⇒ cấu trúc tự lặp. **Sửa:** `children=None` rồi `self.children = children if children is not None else []`.

### Lỗi 4 — Đệ quy quá sâu ở giới hạn mặc định

Với dây xích 1000 tầng, giới hạn đệ quy mặc định (1000) có thể không đủ ⇒ `RecursionError`. Ta tạm hạ giới hạn để minh hoạ và bắt lỗi.

In [26]:
old_lim = sys.getrecursionlimit()
sys.setrecursionlimit(500)
try:
    clone_rec(chain(1000))
    print("không lỗi")
except RecursionError:
    print("RecursionError với giới hạn 500 và cây sâu 1000")
sys.setrecursionlimit(old_lim)
print("bản stack vẫn chạy:", check_deep_copy(chain(1000), clone_stack(chain(1000)))[0])

RecursionError với giới hạn 500 và cây sâu 1000
bản stack vẫn chạy: True


**Sửa:** nâng `sys.setrecursionlimit` (LeetCode thường đã nâng) hoặc dùng Cách 3/4 không đệ quy.

### Lỗi 5 — Quên trường hợp `root is None`

`root.val` trên `None` ⇒ `AttributeError`. Cây rỗng (0 nút) nằm trong ràng buộc ⇒ luôn kiểm tra đầu hàm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Clone **cây**" ⇒ đệ quy một dòng: `Node(x.val, [clone(c) for c in x.children])`.
- "Clone **đồ thị**/có con trỏ phụ" ⇒ thêm `dict` cũ → mới, ghi sổ trước khi đệ quy (LC 133, 138, 1485).
- Bản lặp: stack/queue chứa **cặp** `(cũ, mới)`, gắn con mới ngay khi tạo.
- Kiểm tra deep copy: so `id()` của nút **và** của list `children`, không chỉ so giá trị.
- Python: không bao giờ dùng `[]`/`{}` làm tham số mặc định.

## 15. Biến thể của bài

1. **LC 1485 — Clone Binary Tree With Random Pointer:** có `random` ⇒ bắt buộc `dict`.
2. **LC 133 — Clone Graph:** đồ thị vô hướng có chu trình ⇒ Cách 5.
3. **LC 138 — Copy List with Random Pointer.**
4. **Sao chép kèm biến đổi:** ví dụ nhân đôi giá trị, chỉ giữ các nút có giá trị chẵn, cắt cây ở độ sâu `d` — cùng khung đệ quy, chỉ đổi phần tạo nút (xem Bài 4).
5. **Sao chép DAG giữ nguyên chia sẻ:** nếu hai cha cùng trỏ tới một con, bản sao cũng phải có đúng một con dùng chung ⇒ cần memo (Bài 5).

In [27]:
# Minh hoạ biến thể 4: sao chép nhưng cắt ở độ sâu d (giữ các tầng 0..d)
def clone_depth(root, d):
    if root is None:
        return None
    if d == 0:
        return Node(root.val)
    return Node(root.val, [clone_depth(c, d - 1) for c in root.children])

print(serialize(clone_depth(build_nary(ex2), 1)))
print(serialize(clone_depth(build_nary(ex2), 2)))

[1, None, 2, 3, 4, 5]
[1, None, 2, 3, 4, 5, None, None, 6, 7, None, 8, None, 9, 10]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tuần tự hoá + dựng lại | Mảng trung gian | `O(n)` | `O(n)` | Đã có sẵn hàm serialize/deserialize |
| 2. DFS đệ quy | `Node(val, [clone(c)...])` | `O(n)` | `O(h)` | Ngắn nhất, chuẩn |
| 3. DFS stack | Cặp `(cũ, mới)` trong stack | `O(n)` | `O(n)` | Cây rất sâu |
| 4. BFS | Cặp `(cũ, mới)` trong queue, theo tầng | `O(n)` | `O(w)` | Cần xử lý theo tầng |
| 5. Memo | `dict` cũ → mới | `O(n + E)` | `O(n)` | Đồ thị / DAG / chu trình |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Cây `[1, None, 2, 3, None, 4, None, 5]` có bao nhiêu nút, và 5 là con của nút nào?

### Bài 2 (tính tay)
Với Ví dụ 2, khi chạy Cách 4 (BFS), liệt kê thứ tự các nút được **lấy ra khỏi hàng đợi** (theo giá trị).

### Bài 3 (điền TODO)
Hoàn thiện `max_depth(root)`: độ sâu lớn nhất (số tầng) của cây N-phân, viết **không đệ quy**.

### Bài 4 (tự viết lại)
Viết `clone_map(root, f)`: sao chép cây nhưng giá trị mới là `f(val)`, dùng BFS cặp `(cũ, mới)`.

### Bài 5 (mini challenge)
Cho một **DAG** các `Node` (một nút có thể là con của nhiều nút). Viết `clone_dag(root)` sao cho bản sao **giữ nguyên sự chia sẻ** (hai cha cùng trỏ tới một con ⇒ bản sao cũng vậy), và kiểm tra bằng `is`.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def max_depth(root):
    # TODO: BFS theo tầng hoặc stack lưu (node, depth)
    pass

# Bài 4 — TODO
def clone_map(root, f):
    # TODO: như Cách 4 nhưng tạo Node(f(c.val))
    pass

# Bài 5 — TODO
def clone_dag(root):
    # TODO: dict seen ánh xạ nút cũ -> nút mới
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Con của 1: `2, 3`. (2) Con của 2: `4`; con của 3: `5`. (3) Tổng 5 nút; 5 là con của 3.

**Bài 2.** (1) BFS lấy theo tầng. (2) Tầng 0: 1; tầng 1: 2,3,4,5; tầng 2: 6,7,8,9,10. (3) Tầng 3: 11,12,13; tầng 4: 14.

**Bài 3.** (1) `q = deque([root])`, `depth = 0`. (2) Mỗi vòng xử lý hết `len(q)` nút của tầng hiện tại rồi `depth += 1`. (3) Cây rỗng ⇒ 0.

**Bài 4.** (1) `new_root = Node(f(root.val))`. (2) `new.children = [Node(f(c.val)) for c in old.children]`. (3) `q.extend(zip(...))`.

**Bài 5.** (1) Dùng Cách 5. (2) Kiểm tra: tìm một nút cũ có hai cha, so `is` giữa hai bản sao của con đó trong bản sao. (3) Số nút phân biệt của bản sao = số nút phân biệt của DAG.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
t1 = build_nary([1, None, 2, 3, None, 4, None, 5])
print("số nút:", len(all_nodes(t1)), "| cha của 5:", [x.val for x in all_nodes(t1) if any(c.val == 5 for c in x.children)])
assert len(all_nodes(t1)) == 5 and t1.children[1].children[0].val == 5

# Bài 2
def bfs_pop_order(root):
    order = []
    new_root = Node(root.val)
    q = deque([(root, new_root)])
    while q:
        old, new = q.popleft(); order.append(old.val)
        new.children = [Node(c.val) for c in old.children]
        q.extend(zip(old.children, new.children))
    return order
o2 = bfs_pop_order(build_nary(ex2))
print(o2)
assert o2 == list(range(1, 15))

số nút: 5 | cha của 5: [3]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [30]:
# Bài 3
def max_depth_sol(root):
    if root is None:
        return 0
    q, depth = deque([root]), 0
    while q:
        for _ in range(len(q)):
            x = q.popleft(); q.extend(x.children)
        depth += 1
    return depth

assert max_depth_sol(None) == 0
assert max_depth_sol(build_nary(ex1)) == 3
assert max_depth_sol(build_nary(ex2)) == 5
assert max_depth_sol(chain(1000)) == 1000

# Bài 4
def clone_map_sol(root, f):
    if root is None:
        return None
    new_root = Node(f(root.val))
    q = deque([(root, new_root)])
    while q:
        old, new = q.popleft()
        new.children = [Node(f(c.val)) for c in old.children]
        q.extend(zip(old.children, new.children))
    return new_root

doubled = clone_map_sol(build_nary(ex1), lambda v: v * 2)
assert serialize(doubled) == [2, None, 6, 4, 8, None, 10, 12]
rng = random.Random(4)
for _ in range(200):
    orig = random_nary(rng, rng.randint(0, 40))
    same = clone_map_sol(orig, lambda v: v)
    assert check_deep_copy(orig, same)[0]
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
# Bài 5
def clone_dag_sol(root):
    seen = {}
    def dfs(old):
        if old in seen:
            return seen[old]
        new = Node(old.val)
        seen[old] = new
        new.children = [dfs(c) for c in old.children]
        return new
    return dfs(root) if root else None

shared = Node("s", [Node("leaf")])
dag = Node("r", [Node("a", [shared]), Node("b", [shared])])
cp = clone_dag_sol(dag)
sa, sb = cp.children[0].children[0], cp.children[1].children[0]
print("bản sao giữ chia sẻ?", sa is sb, "| là nút mới?", sa is not shared)
assert sa is sb and sa is not shared and sa.children[0] is not shared.children[0]
# so với đệ quy thuần: tách thành hai nút riêng
cp_rec = clone_rec(dag)
assert cp_rec.children[0].children[0] is not cp_rec.children[1].children[0]
# trên cây thường, clone_dag cũng là bản sao sâu hợp lệ
assert check_deep_copy(build_nary(ex2), clone_dag_sol(build_nary(ex2)))[0]
print("Đáp án Bài 1–5: mọi assert đều qua")

bản sao giữ chia sẻ? True | là nút mới? True
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template đệ quy (cây):**

```python
def cloneTree(self, root):
    if root is None:
        return None
    return Node(root.val, [self.cloneTree(c) for c in root.children])
```

**Template lặp (cặp cũ/mới):**

```python
new_root = Node(root.val)
q = deque([(root, new_root)])
while q:
    old, new = q.popleft()
    new.children = [Node(c.val) for c in old.children]
    q.extend(zip(old.children, new.children))
```

**Template đồ thị:** thêm `seen = {}`; `if old in seen: return seen[old]`; ghi `seen[old] = new` **trước** khi xử lý con.

**Checklist trước khi submit:**
- [ ] `root is None` ⇒ trả `None`.
- [ ] `children` của bản sao là **list mới** chứa **nút mới** (không `root.children`).
- [ ] Giữ đúng **thứ tự** con.
- [ ] Không dùng `children=[]` làm tham số mặc định khi tự định nghĩa lớp.
- [ ] Cây sâu 1000: nâng recursion limit hoặc dùng bản lặp.
- [ ] Đúng chữ ký: `cloneTree(self, root: 'Node') -> 'Node'`.